## Imports

Minimal imports for inspecting the provided `data/` files.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd


## Data Preview

Resolves the project `data/` directory (even if the notebook is run from `data_processing/`) and prints:
- column names
- pandas-inferred dtypes
- first 5 rows
- per-cell Python types for the preview
- null counts

This is for approval/inspection only (read-only).

## Assumptions / Notes

- Path resolution: `DATA_DIR` is found by walking up from the current working directory until a `data/` folder is found (assumes exactly one project `data/`).
- This notebook is **inspection-only**: it prints pandas-inferred dtypes and a small preview, but does not enforce a schema contract.
- `pd.read_csv(path)` loads the full CSV into memory; for `transactions.csv` this can be slow/high-memory. If needed, we can switch previews to chunked reads.
- Pandas dtypes are **inferred** and may not match the desired cleaned types (e.g., currency strings). Approval here is about raw file shape/content.


In [2]:
def find_data_dir(start: Path | None = None) -> Path:
    """Find the repository's data/ directory regardless of notebook working directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        data_dir = candidate / "data"
        if data_dir.is_dir():
            return data_dir
    raise FileNotFoundError(
        "Could not locate a 'data/' directory from the current working directory. "
        "Run this notebook from within the project."
    )


DATA_DIR = find_data_dir()
print("Resolved DATA_DIR:", DATA_DIR)

FILES = {
    "transactions": DATA_DIR / "transactions.csv",
    "cards": DATA_DIR / "cards.csv",
    "users": DATA_DIR / "users.csv",
    "mcc_codes": DATA_DIR / "mcc_codes.json",
}


def preview_csv(path: Path, *, nrows: int = 5) -> None:
    print(f"\n=== {path.name} ===")
    df = pd.read_csv(path)
    print("Columns:")
    print(list(df.columns))
    print("\nDtypes (pandas-inferred):")
    print(df.dtypes)
    print("\nHead:")
    display(df.head(nrows))
    print("\nPer-cell Python types for head (for quick inspection):")
    display(df.head(nrows).applymap(lambda v: type(v).__name__))
    print("\nNull counts:")
    print(df.isna().sum().sort_values(ascending=False))


for name, path in FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    if path.suffix.lower() == ".csv":
        preview_csv(path)
    else:
        import json

        print(f"\n=== {path.name} ===")
        obj = json.loads(path.read_text(encoding="utf-8"))
        print("Top-level type:", type(obj).__name__)
        if isinstance(obj, dict):
            print("Example items (first 10):")
            for k, v in list(obj.items())[:10]:
                print(f"- {k!r}: {v!r}")


Resolved DATA_DIR: /Users/nicholasp/Personal Coding/JHU/personal finance/data

=== transactions.csv ===
Columns:
['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors']

Dtypes (pandas-inferred):
id                  int64
date               object
client_id           int64
card_id             int64
amount             object
use_chip           object
merchant_id         int64
merchant_city      object
merchant_state     object
zip               float64
mcc                 int64
errors             object
dtype: object

Head:


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475333,2010-01-01 00:07:00,1807,165,$4.81,Swipe Transaction,20519,Bronx,NY,10464.0,5942,NaN
1,7475367,2010-01-01 01:06:00,1758,4686,$87.09,Online Transaction,17976,ONLINE,NaN,NaN,4900,NaN
2,7475373,2010-01-01 01:11:00,1642,4281,$3.96,Swipe Transaction,44625,Manasquan,NJ,8736.0,5812,NaN
3,7475404,2010-01-01 01:56:00,1642,4281,$3.35,Swipe Transaction,44625,Manasquan,NJ,8736.0,5812,NaN
4,7475414,2010-01-01 02:08:00,1084,3454,$6.63,Swipe Transaction,45926,Tacoma,WA,98403.0,5814,NaN



Per-cell Python types for head (for quick inspection):


/var/folders/36/y0l5p2td2rz6f927knpzb0smcvr2cp/T/ipykernel_67420/280170681.py:35: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  display(df.head(nrows).applymap(lambda v: type(v).__name__))


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,int,str,int,int,str,str,int,str,str,float,int,float
1,int,str,int,int,str,str,int,str,float,float,int,float
2,int,str,int,int,str,str,int,str,str,float,int,float
3,int,str,int,int,str,str,int,str,str,float,int,float
4,int,str,int,int,str,str,int,str,str,float,int,float


...

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,monthly_discretionary_limits
0,140,37,72,1983,2,Female,108 Washington Street,36.28,-86.83,"$20,657","$42,120","$72,801",739,4,"$1,755"
1,838,26,70,1993,7,Male,349 First Drive,33.78,-84.44,"$24,382","$49,713","$91,776",763,1,"$2,071"
2,1219,28,72,1991,6,Female,9977 Oak Avenue,38.68,-76.17,"$28,497","$58,105","$70,254",688,3,"$2,421"
3,348,49,70,1970,7,Female,480 Seventh Lane,40.22,-74.76,"$22,747","$46,377","$79,980",719,3,"$1,932"
4,633,36,69,1983,10,Female,5506 Fifth Boulevard,33.88,-118.27,"$24,611","$50,179","$110,515",743,1,"$2,091"



Per-cell Python types for head (for quick inspection):


/var/folders/36/y0l5p2td2rz6f927knpzb0smcvr2cp/T/ipykernel_67420/280170681.py:35: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  display(df.head(nrows).applymap(lambda v: type(v).__name__))


,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,monthly_discretionary_limits
0,int,int,int,int,int,str,str,float,float,str,str,str,int,int,str
1,int,int,int,int,int,str,str,float,float,str,str,str,int,int,str
2,int,int,int,int,int,str,str,float,float,str,str,str,int,int,str
3,int,int,int,int,int,str,str,float,float,str,str,str,int,int,str
4,int,int,int,int,int,str,str,float,float,str,str,str,int,int,str



Null counts:
id                              0
current_age                     0
retirement_age                  0
birth_year                      0
birth_month                     0
gender                          0
address                         0
latitude                        0
longitude                       0
per_capita_income               0
yearly_income                   0
total_debt                      0
credit_score                    0
num_credit_cards                0
monthly_discretionary_limits    0
dtype: int64

=== mcc_codes.json ===
Top-level type: dict
Example items (first 10):
- '5812': 'Eating Places and Restaurants'
- '5541': 'Service Stations'
- '7996': 'Amusement Parks, Carnivals, Circuses'
- '5411': 'Grocery Stores, Supermarkets'
- '4784': 'Tolls and Bridge Fees'
- '4900': 'Utilities - Electric, Gas, Water, Sanitary'
- '5942': 'Book Stores'
- '5814': 'Fast Food Restaurants'
- '4829': 'Money Transfer'
- '5311': 'Department Stores'
